## QML-FAST: 3-field example

Demonstrates the full QML pipeline on 3 correlated Gaussian fields at nside=16:
1. Build pixel covariance, mask, mode deprojection
2. Compute Fisher matrix with `getF`
3. Run MC simulations and obtain QML estimates with `get_y`
4. Compare analytic and empirical error bars

In [ ]:
import sys
sys.path.append('./')
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import opt_einsum as oe
import scipy as sp
from tqdm import tqdm

from utilities import *
from qmlfast import *

### 1. Simulation setup

In [ ]:
nside = 16
lmax  = 3*nside - 1
ell   = np.arange(lmax+1).astype(float)

S_aa = (ell+1)**(-2)
S_bb = 3*(ell+1)**(-2)
S_cc = 5*(ell+1)**(-2)
S_ab = np.sqrt(S_aa*S_bb)
S_ca = np.sqrt(S_aa*S_cc)
S_bc = np.sqrt(S_bb*S_cc)

N_a, N_b, N_c = 5e-2, 5e-2, 1e-2
A_ab, A_ca, A_bc = 0.3, 0.5, 0.5

C_aa = S_aa + N_a
C_bb = S_bb + N_b
C_cc = S_cc + N_c
C_ab = A_ab*S_ab
C_ca = A_ca*S_ca
C_bc = A_bc*S_bc

In [ ]:
# Mask
vec = hp.ang2vec(np.pi / 2, np.pi/2)
ipix_disc = hp.query_disc(nside=nside, vec=vec, radius=np.radians(51))
vec1 = hp.ang2vec(np.pi / 3, 3*np.pi/2)
ipix_disc1 = hp.query_disc(nside=nside, vec=vec1, radius=np.radians(47))

m = np.ones(hp.nside2npix(nside))
m[ipix_disc] = 0.
m[ipix_disc1] = 0.
hp.mollview(m, title='Mask')
print(f'Sky fraction: {m.mean():.2f}')

In [ ]:
def get_pix_cov_block(cl, Pl_ij, lmin, lmax):
    return oe.contract('ijk,i->jk', Pl_ij[lmin:lmax], cl[lmin:lmax])

theta, phi = theta_phi(nside)
theta = theta[m==1]
phi = phi[m==1]
Pl_ij = get_Pl_ij(theta, phi, nside, lmax=3*nside-1)

Np = len(theta)
omega_pix = 4 * np.pi / hp.nside2npix(nside)

large_cov = np.zeros([3*Np, 3*Np])
large_cov[block_np(0, 0, Np)] = get_pix_cov_block(S_aa, Pl_ij, 0, 3*nside)  # FIX: was 3*nside-1 + np.eye(Np)*N_a/omega_pix
large_cov[block_np(0, 1, Np)] = get_pix_cov_block(S_ab, Pl_ij, 0, 3*nside)  # FIX: was 3*nside-1
large_cov[block_np(0, 2, Np)] = get_pix_cov_block(S_ca, Pl_ij, 0, 3*nside)  # FIX: was 3*nside-1
large_cov[block_np(1, 0, Np)] = large_cov[block_np(0, 1, Np)].T
large_cov[block_np(1, 1, Np)] = get_pix_cov_block(S_bb, Pl_ij, 0, 3*nside)  # FIX: was 3*nside-1 + np.eye(Np)*N_b/omega_pix
large_cov[block_np(1, 2, Np)] = get_pix_cov_block(S_bc, Pl_ij, 0, 3*nside)  # FIX: was 3*nside-1
large_cov[block_np(2, 0, Np)] = large_cov[block_np(0, 2, Np)].T
large_cov[block_np(2, 1, Np)] = large_cov[block_np(1, 2, Np)].T
large_cov[block_np(2, 2, Np)] = get_pix_cov_block(S_cc, Pl_ij, 0, 3*nside)  # FIX: was 3*nside-1 + np.eye(Np)*N_c/omega_pix

### 2. Mode deprojection ($\ell < \ell_{\min}$)

In [ ]:
ell0 = 4
Z, pi = construct_Z_and_pi(theta, phi, ell0=ell0, lmax=3*nside-1)

for i in range(3):
    for j in range(3):
        block = large_cov[block_np(i,j,Np)]
        large_cov[block_np(i,j,Np)] = pi@block@pi.T

eta = 1e-1
M = np.linalg.inv(large_cov + eta*sp.linalg.block_diag(Z@Z.T, Z@Z.T, Z@Z.T))
for i in range(3):
    for j in range(3):
        block = M[block_np(i,j,Np)]
        M[block_np(i,j,Np)] = pi@block@pi.T

### 3. Fisher matrix

`getF` uses packed storage internally (no zero-padding). For memory-constrained problems, pass `budget=N` to batch the computation.

In [ ]:
Nf = 3
F_idx = np.array([(i, j, l) for l in range(0, 3*nside)
                  for i in range(Nf) for j in range(i, Nf)])
C_map = np.ones([3, 3])

Y_r_all = sph_harm_y_real_all(3*nside, theta, phi)

F = getF(Y_r_all, M, F_idx, Nf, Np, C_map)
print(f'F shape: {F.shape}')

In [ ]:
# Sanity check: F elements with ell < ell_min should be 0
null_block = F[np.ix_(F_idx[:,2]<ell0, F_idx[:,2]<ell0)]
print(f'F[ell<{ell0}] max = {np.max(np.abs(null_block)):.1e}  (should be ~0)')

### 4. Run simulations and compute QML estimates

For simulation loops, pre-pack the basis and use `get_y_packed` directly to avoid repacking each call.

In [ ]:
# Pre-pack for the simulation loop
V_packed, offsets, ranks = pack_sph_harm(Y_r_all)

ys = []
for i in tqdm(range(2500)):
    alm_a, alm_b, alm_c = hp.synalm([C_aa, C_ab, C_ca, C_bb, C_bc, C_cc], lmax=lmax)
    x = np.stack([hp.alm2map(alm_a, nside)[m==1],
                  hp.alm2map(alm_b, nside)[m==1],
                  hp.alm2map(alm_c, nside)[m==1]])
    y = get_y_packed(x, V_packed, offsets, ranks, M, F_idx, Nf, Np)
    ys.append(y)

In [ ]:
# Remove null block and solve
ys = np.array(ys)[:, F_idx[:,2]>=ell0]
F = F[np.ix_(F_idx[:,2]>=ell0, F_idx[:,2]>=ell0)]
invF = np.linalg.inv(F)
es = ys @ invF
e_mean = es.mean(axis=0)
err_analytic = invF
err_empirical = np.cov(es, rowvar=False, ddof=0)
f_idx = F_idx[F_idx[:,2]>=ell0]

### 5. Plot auto-power spectra

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(12, 8))

spectra = [
    ((0,0), C_aa, r'$C_{aa}$'),
    ((1,1), C_bb, r'$C_{bb}$'),
    ((2,2), C_cc, r'$C_{cc}$'),
]

for col, ((fi, fj), C_true, label) in enumerate(spectra):
    idx = (f_idx[:,0]==fi) * (f_idx[:,1]==fj)
    ells_plot = np.arange(ell0, 3*nside)

    axs[0, col].errorbar(ells_plot, e_mean[idx],
                         yerr=np.sqrt(err_analytic[idx, idx]), lw=1, label='QML')
    axs[0, col].semilogy(np.arange(len(C_true)), C_true, '.', label='True')
    axs[0, col].set_xlabel(r'$\ell$', fontsize=14)
    axs[0, col].set_ylabel(label, fontsize=14)
    axs[0, col].legend(fontsize=12)
    axs[0, col].grid(alpha=0.5)

    axs[1, col].semilogy(ells_plot, np.sqrt(err_analytic[idx, idx]), lw=1, label='Analytic')
    axs[1, col].semilogy(ells_plot, np.sqrt(err_empirical[idx, idx]), lw=1, label='Empirical')
    axs[1, col].set_xlabel(r'$\ell$', fontsize=14)
    axs[1, col].set_ylabel(f'$\\sigma$({label})', fontsize=14)
    axs[1, col].legend(fontsize=12)
    axs[1, col].grid(alpha=0.5)

plt.tight_layout()
plt.show()

### 6. Plot cross-power spectra

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(12, 8))

cross = [
    ((0,1), C_ab, r'$C_{ab}$'),
    ((0,2), C_ca, r'$C_{ac}$'),
    ((1,2), C_bc, r'$C_{bc}$'),
]

for col, ((fi, fj), C_true, label) in enumerate(cross):
    idx = (f_idx[:,0]==fi) * (f_idx[:,1]==fj)
    ells_plot = np.arange(ell0, 3*nside)

    axs[0, col].errorbar(ells_plot, e_mean[idx],
                         yerr=np.sqrt(err_analytic[idx, idx]), lw=1, label='QML')
    axs[0, col].plot(np.arange(len(C_true)), C_true, '.', label='True')
    axs[0, col].set_xlabel(r'$\ell$', fontsize=14)
    axs[0, col].set_ylabel(label, fontsize=14)
    axs[0, col].legend(fontsize=12)
    axs[0, col].grid(alpha=0.5)

    axs[1, col].semilogy(ells_plot, np.sqrt(err_analytic[idx, idx]), lw=1, label='Analytic')
    axs[1, col].semilogy(ells_plot, np.sqrt(err_empirical[idx, idx]), lw=1, label='Empirical')
    axs[1, col].set_xlabel(r'$\ell$', fontsize=14)
    axs[1, col].set_ylabel(f'$\\sigma$({label})', fontsize=14)
    axs[1, col].legend(fontsize=12)
    axs[1, col].grid(alpha=0.5)

plt.tight_layout()
plt.show()